In [1]:
import os
import numpy as np
import pybamm
import tqdm

In [2]:
%cd ..

/home/dlu-apa/repositories/foo_cleaned/src


/home/dlu-apa/.pyenv/versions/3.11.4/lib/python3.11/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [3]:
spm = pybamm.lithium_ion.SPM()
params_bat = pybamm.ParameterValues("Prada2013")
spm.events = []

In [4]:
family = "GRF"
N_total = 33000
data = np.load(f"../data/{family}_{N_total}.npz")
random_seed = 7
ratio = 0.01

In [5]:
from util.FNO_util import train_test_split

In [6]:
_, subset_data = train_test_split(data, N_total=N_total, test_ratio=ratio, seed=random_seed)

In [7]:

func_I = np.array(subset_data["current"])

### Anode data ###
cn_anode = np.array(subset_data["cn_anode"])
c0_anode = np.array(subset_data["c0_anode"])
D_anode = np.array(subset_data["Dan"])

###Cathode data ###
cn_cathode = np.array(subset_data["cn_cathode"])
c0_cathode = np.array(subset_data["c0_cathode"])
D_cathode = np.array(subset_data["Dca"])

soc = np.array(subset_data["soc"])

In [8]:
def single_run(t, I_func, Dan, Dca, soc):
    params_local = pybamm.ParameterValues("Prada2013")
    params_local["Current function [A]"] = pybamm.Interpolant(t, -1. * I_func, pybamm.t)
    params_local["Negative particle diffusivity [m2.s-1]"] = 10 ** Dan
    params_local["Positive particle diffusivity [m2.s-1]"] = 10 ** Dca
    sim = pybamm.Simulation(spm, parameter_values=params_local)
    sol = sim.solve(initial_soc=soc, t_eval=t)

    c0_anode = sol["Negative particle concentration"].entries[:, 0, 0]
    cn_anode = sol["Negative particle concentration"].entries[:, 0, :]
    c0_cathode = sol["Positive particle concentration"].entries[:, 0, 0]
    cn_cathode = sol["Positive particle concentration"].entries[:, 0, :]

    return cn_anode, c0_anode, cn_cathode, c0_cathode

In [9]:
def compare_results(sim_cn_anode, sim_c0_anode, sim_cn_cathode, sim_c0_cathode, data_cn_anode, data_c0_anode, data_cn_cathode, data_c0_cathode):
    assert np.allclose(sim_cn_anode, data_cn_anode)
    assert np.allclose(sim_c0_anode, data_c0_anode)
    assert np.allclose(sim_cn_cathode, data_cn_cathode)
    assert np.allclose(sim_c0_cathode, data_c0_cathode)


In [11]:
pbar = tqdm.trange(len(func_I), desc="Comparing with data")

for i in pbar:
    t_eval = np.linspace(0, 3600, 75)

    sim_cn_anode, sim_c0_anode, sim_cn_cathode, sim_c0_cathode = single_run(t_eval, func_I[i], D_anode[i], D_cathode[i], soc[i])
    compare_results(sim_cn_anode, sim_c0_anode, sim_cn_cathode, sim_c0_cathode, cn_anode[i], c0_anode[i], cn_cathode[i], c0_cathode[i])


Comparing with data: 100%|██████████| 330/330 [01:00<00:00,  5.44it/s]
